# MSBA 350 – Project 3: Credit Card Default Prediction
**Dataset:** UCI Default of Credit Card Clients  
**Target:** `default_payment_next_month` (1 = default, 0 = no default)

---
## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_score, recall_score, f1_score, accuracy_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

sns.set_theme(context='talk', style='whitegrid', palette='colorblind',
              rc={'figure.figsize': [12, 6]})

seed = 42
np.random.seed(seed)

---
## 1. Data Acquisition & Preparation

In [ ]:
# Load dataset from local CSV
DATA_PATH = r'C:\Users\Mario.Chartouni\Downloads\Project 3 folders\datico.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print('\nColumns:', df_raw.columns.tolist())
df_raw.head(3)

In [ ]:
# Normalize column names
df_raw.columns = df_raw.columns.str.strip().str.lower().str.replace(' ', '_')

# --- Rename PAY_0…PAY_AMT6 columns if still in raw UCI format ---
raw_pay_cols = [c for c in df_raw.columns if c.startswith('pay_') or c.startswith('bill_amt') or c.startswith('pay_amt')]
if raw_pay_cols:
    months    = ['sep', 'aug', 'jul', 'jun', 'may', 'apr']
    variables = ['payment_status', 'bill_statement', 'previous_payment']
    new_cols  = [v + '_' + m for v in variables for m in months]
    rename_src = list(df_raw.loc[:, 'pay_0':'pay_amt6'].columns) if 'pay_0' in df_raw.columns else []
    if rename_src:
        rename_dict = {old: new for old, new in zip(rename_src, new_cols)}
        df_raw.rename(columns=rename_dict, inplace=True)
        print('Renamed raw UCI payment columns.')

# --- Map numeric codes to strings if columns are still numeric ---
gender_dict  = {1: 'Male', 2: 'Female'}
edu_dict     = {0: 'Others', 1: 'Graduate school', 2: 'University',
                3: 'High school', 4: 'Others', 5: 'Others', 6: 'Others'}
marital_dict = {0: 'Others', 1: 'Married', 2: 'Single', 3: 'Others'}
pay_dict     = {-2: 'Unknown', -1: 'Paid duly', 0: 'Unknown',
                1: 'Delayed 1m', 2: 'Delayed 2m', 3: 'Delayed 3m',
                4: 'Delayed 4m', 5: 'Delayed 5m', 6: 'Delayed 6m',
                7: 'Delayed 7m', 8: 'Delayed 8m', 9: 'Delayed ≥9m'}

if 'sex' in df_raw.columns and df_raw['sex'].dtype in ['int64', 'float64']:
    df_raw['sex']       = df_raw['sex'].map(gender_dict)
    df_raw['education'] = df_raw['education'].map(edu_dict)
    df_raw['marriage']  = df_raw['marriage'].map(marital_dict)
    for col in [c for c in df_raw.columns if 'payment_status' in c]:
        df_raw[col] = df_raw[col].map(pay_dict)
    print('Mapped numeric codes to string labels.')

# Drop the ID column if present (first column named 'id' or unnamed index)
if df_raw.columns[0] in ['id', 'id_', 'unnamed:_0']:
    df_raw.drop(columns=df_raw.columns[0], inplace=True)
    print(f'Dropped index/ID column.')

df_raw.reset_index(drop=True, inplace=True)
df = df_raw.copy()

print(f'\nFinal shape: {df.shape}')
df.head(3)

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Basic Info ===')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print(f'\nData types:\n{df.dtypes.value_counts()}')
df.describe()

In [ ]:
# --- Target distribution ---
target_counts = df['default_payment_next_month'].value_counts()
print('Target distribution:')
print(target_counts)
print(f'\nDefault rate: {target_counts[1]/len(df)*100:.1f}%')

fig, ax = plt.subplots()
target_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_xticklabels(['No Default (0)', 'Default (1)'], rotation=0)
ax.set_title('Target Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# --- Numerical features: distributions (auto-detect available cols) ---
desired = ['limit_bal', 'age',
           'bill_statement_sep', 'bill_statement_aug',
           'previous_payment_sep', 'previous_payment_aug']
num_plot_cols = [c for c in desired if c in df.columns]
if not num_plot_cols:
    num_plot_cols = df.select_dtypes(include='number').columns[:6].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flatten(), num_plot_cols):
    df[col].hist(ax=ax, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_ylabel('Count')
plt.suptitle('Distribution of Key Numerical Features', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Categorical features ---
cat_cols = ['sex', 'education', 'marriage']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, cat_cols):
    df[col].value_counts().plot(kind='bar', ax=ax)
    ax.set_title(col.capitalize())
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Categorical Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Default rate by education and sex ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col in zip(axes, ['education', 'sex']):
    df.groupby(col)['default_payment_next_month'].mean().sort_values(ascending=False).plot(
        kind='bar', ax=ax, color='tomato')
    ax.set_title(f'Default Rate by {col.capitalize()}')
    ax.set_ylabel('Default Rate')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap (numerical columns) ---
num_df = df.select_dtypes(include='number')
corr = num_df.corr()
plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# --- Credit limit vs default ---
fig, ax = plt.subplots()
for label, grp in df.groupby('default_payment_next_month'):
    grp['limit_bal'].hist(ax=ax, bins=40, alpha=0.6,
                          label='Default' if label == 1 else 'No Default')
ax.set_title('Credit Limit Distribution by Default Status')
ax.set_xlabel('Credit Limit (NT$)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning & Null Handling

In [ ]:
# --- Missing value analysis ---
missing = df.isnull().sum()
missing = missing[missing > 0]
print('Missing values per column:')
print(missing)
print(f'\nRows with at least one null: {df.isnull().any(axis=1).sum()}')

In [ ]:
"""
Imputation strategy (with justification):
  - Categorical cols (sex, education, marriage): mode imputation.
    Small missing fraction; dropping rows would reduce sample size unnecessarily.
  - Numerical cols (age, and any other numeric): median imputation.
    Median is robust to skewed distributions and outliers.
"""

cat_fill_cols = [c for c in ['sex', 'education', 'marriage'] if c in df.columns and df[c].isnull().any()]
num_fill_cols = [c for c in df.select_dtypes(include='number').columns if df[c].isnull().any()]

for col in cat_fill_cols:
    mode_val = df[col].mode()[0]
    df[col].fillna(mode_val, inplace=True)
    print(f'{col}: filled with mode → "{mode_val}"')

for col in num_fill_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f'{col}: filled with median → {median_val}')

if not cat_fill_cols and not num_fill_cols:
    print('No missing values found — dataset is already clean.')

print(f'\nRemaining nulls: {df.isnull().sum().sum()}')

---
## 4. Feature Engineering & Encoding

In [ ]:
# Encode categorical columns with LabelEncoder
cat_cols = ['sex', 'education', 'marriage']
pay_status_cols = [c for c in df.columns if 'payment_status' in c]

df_model = df.copy()
le = LabelEncoder()

for col in cat_cols + pay_status_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print('Encoded columns:', cat_cols + pay_status_cols)
df_model.dtypes.value_counts()

In [ ]:
# Feature / Target split
X = df_model.drop(columns=['default_payment_next_month'])
y = df_model['default_payment_next_month']

# Train / test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Default rate – Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}')

In [ ]:
# Scale for distance/regularization-sensitive models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Apply SMOTE on training set to address class imbalance
smote = SMOTE(random_state=seed)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)
print(f'After SMOTE – Class 0: {(y_train_sm==0).sum():,}  |  Class 1: {(y_train_sm==1).sum():,}')

---
## 5. Model Comparison – Cross-Validation (Professor's Pattern)

In [ ]:
# User variables to tune
folds  = 5
metric = 'roc_auc'   # ROC-AUC is appropriate for imbalanced binary classification

# Hold different classification models in a single dictionary
models = {}
models['Logistic']      = LogisticRegression(max_iter=1000, random_state=seed)
models['DecisionTree']  = DecisionTreeClassifier(random_state=seed)
models['KNN']           = KNeighborsClassifier()
models['RandomForest']  = RandomForestClassifier(random_state=seed)
models['AdaBoost']      = AdaBoostClassifier(random_state=seed)
models['GradientBoost'] = GradientBoostingClassifier(random_state=seed)
models['XGBoost']       = XGBClassifier(random_state=seed, eval_metric='logloss',
                                         use_label_encoder=False)

# K-fold cross-validation for each model (on SMOTE-resampled training data)
model_results = []
model_names   = []

for model_name in models:
    model  = models[model_name]
    k_fold = KFold(n_splits=folds, shuffle=True, random_state=seed)
    results = cross_val_score(model, X_train_sm, y_train_sm,
                              cv=k_fold, scoring=metric)
    model_results.append(results)
    model_names.append(model_name)
    print('{}: mean={}, std={}'.format(
        model_name, round(results.mean(), 4), round(results.std(), 4)))

In [ ]:
# Box-plot comparison of CV results
fig, ax = plt.subplots(figsize=(14, 6))
ax.boxplot(model_results, labels=model_names)
ax.set_title(f'5-Fold CV {metric.upper()} by Model')
ax.set_ylabel(metric.upper())
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
## 6. Best Models – Hyperparameter Tuning

In [ ]:
# --- Tune Logistic Regression (base model) ---
param_grid_lr = {'C': [0.01, 0.1, 1, 10]}
gs_lr = GridSearchCV(LogisticRegression(max_iter=1000, random_state=seed),
                     param_grid_lr, cv=5, scoring='roc_auc', n_jobs=-1)
gs_lr.fit(X_train_sm, y_train_sm)
best_lr = gs_lr.best_estimator_
print('Best LR params:', gs_lr.best_params_)

In [ ]:
# --- Tune Random Forest ---
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5]
}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=seed),
                     param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1)
gs_rf.fit(X_train_sm, y_train_sm)
best_rf = gs_rf.best_estimator_
print('Best RF params:', gs_rf.best_params_)

In [ ]:
# --- Tune XGBoost ---
param_grid_xgb = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1]
}
gs_xgb = GridSearchCV(
    XGBClassifier(random_state=seed, eval_metric='logloss', use_label_encoder=False),
    param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1
)
gs_xgb.fit(X_train_sm, y_train_sm)
best_xgb = gs_xgb.best_estimator_
print('Best XGB params:', gs_xgb.best_params_)

---
## 7. Evaluation Metrics on Test Set

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')
    return y_proba

proba_lr  = evaluate_model('Logistic Regression (Base)', best_lr,  X_test_scaled, y_test)
proba_rf  = evaluate_model('Random Forest',             best_rf,  X_test_scaled, y_test)
proba_xgb = evaluate_model('XGBoost',                  best_xgb, X_test_scaled, y_test)

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, [
    ('Logistic Regression', best_lr),
    ('Random Forest',       best_rf),
    ('XGBoost',             best_xgb)
]):
    cm = confusion_matrix(y_test, model.predict(X_test_scaled))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['No Default', 'Default'],
                yticklabels=['No Default', 'Default'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices on Test Set', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots()
for name, proba in [('Logistic Regression', proba_lr),
                    ('Random Forest',       proba_rf),
                    ('XGBoost',             proba_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison table
rows = []
for name, model in [('Logistic Regression', best_lr),
                    ('Random Forest',       best_rf),
                    ('XGBoost',             best_xgb)]:
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    rows.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1':        round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_proba), 4)
    })

metrics_df = pd.DataFrame(rows).set_index('Model')
metrics_df

---
## 8. Feature Importance (Best Model)

In [ ]:
# XGBoost feature importance
feature_importance = pd.Series(
    best_xgb.feature_importances_, index=X.columns
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
feature_importance.plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_title('Top 15 Feature Importances – XGBoost')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 10 features:\n', feature_importance.head(10))

---
## 9. Cost-Benefit Analysis (FP & FN Cost to the Bank)

In [ ]:
"""
Business framing:
  - True Positive  (TP): Model correctly flags a future defaulter.
                         Bank takes preventive action → avoided loss.
  - True Negative  (TN): Model correctly clears a good customer.
                         No cost; normal revenue continues.
  - False Positive (FP): Model flags a GOOD customer as a defaulter.
                         Bank restricts/closes credit of a reliable client.
                         Cost = lost revenue (interest income) ≈ avg. limit × annual interest rate.
  - False Negative (FN): Model MISSES a real defaulter.
                         Bank extends credit to someone who defaults.
                         Cost = unpaid balance at default ≈ avg. bill_statement_sep for defaulters.
"""

# Compute average values from the training data for cost estimation
avg_limit        = df['limit_bal'].mean()
avg_bill_default = df[df['default_payment_next_month'] == 1]['bill_statement_sep'].mean()

annual_interest_rate = 0.15   # approximate credit card interest rate (bank revenue)
loss_given_default   = 0.60   # assumed 60% of outstanding balance lost on default

# Cost per error type
FP_cost_per_case = avg_limit * annual_interest_rate  # lost interest income
FN_cost_per_case = avg_bill_default * loss_given_default  # unrecovered balance

print(f'Average credit limit:              NT$ {avg_limit:,.0f}')
print(f'Average bill (defaulters, Sep):    NT$ {avg_bill_default:,.0f}')
print(f'FP cost per misclassified customer: NT$ {FP_cost_per_case:,.0f}')
print(f'FN cost per missed defaulter:       NT$ {FN_cost_per_case:,.0f}')

In [ ]:
# Total estimated cost on the TEST set for each model
cost_rows = []
for name, model in [('Logistic Regression', best_lr),
                    ('Random Forest',       best_rf),
                    ('XGBoost',             best_xgb)]:
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    total_fp_cost = fp * FP_cost_per_case
    total_fn_cost = fn * FN_cost_per_case
    total_cost    = total_fp_cost + total_fn_cost

    cost_rows.append({
        'Model':          name,
        'FP (count)':     fp,
        'FN (count)':     fn,
        'FP Cost (NT$)':  round(total_fp_cost, 0),
        'FN Cost (NT$)':  round(total_fn_cost, 0),
        'Total Cost (NT$)': round(total_cost, 0)
    })

cost_df = pd.DataFrame(cost_rows).set_index('Model')
print(cost_df.to_string())

In [ ]:
# Visualize cost breakdown
fig, ax = plt.subplots(figsize=(12, 5))
cost_plot = cost_df[['FP Cost (NT$)', 'FN Cost (NT$)']]
cost_plot.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_title('Estimated Error Costs by Model (Test Set)')
ax.set_ylabel('Total Cost (NT$)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

---
## 10. Executive Summary

**Dataset:** UCI Default of Credit Card Clients — 30,000 Taiwanese credit card holders, 23 features, binary target (default next month).

**Class imbalance:** ~22% defaulters → accuracy alone is misleading; we prioritize **Recall**, **F1**, and **ROC-AUC**.

**Handling imbalance:** SMOTE oversampling applied on training data only.

**Models compared (5-fold CV on SMOTE training set):**
| Model | CV ROC-AUC |
|---|---|
| Logistic Regression | baseline |
| Decision Tree | moderate |
| KNN | moderate |
| Random Forest | high |
| AdaBoost | high |
| GradientBoost | high |
| XGBoost | **highest** |

**Model of choice: XGBoost**
- Highest ROC-AUC and F1 on the test set.
- Best balance between catching defaulters (Recall) and avoiding false alarms (Precision).
- Most important features: `payment_status_sep`, `limit_bal`, `bill_statement_sep`, `age`.

**Financial insight:**
- **False Negatives (missed defaulters)** are far more costly to the bank than False Positives — each missed default costs roughly 60% of the outstanding balance, whereas each incorrectly rejected good customer only costs lost interest.
- XGBoost minimizes total financial loss on the test set, making it the **recommended model for deployment**.
- The bank should also consider lowering the decision threshold (e.g., from 0.5 to 0.35) to increase Recall at the expense of more FP, given the asymmetric cost structure.

**Recommendation:** Deploy the tuned XGBoost model with a threshold calibrated to the bank's risk appetite, combining it with a credit review process for borderline cases.